In [2]:
#Data Preparation & Pipeline Lead

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pickle
import os

In [3]:
df = pd.read_csv('../data/Diabetes_and_LifeStyle_Dataset_.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head()

Dataset Shape: (97297, 31)

Columns: ['Age', 'gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c', 'diabetes_risk_score', 'diabetes_stage', 'diagnosed_diabetes']


,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,52,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


In [5]:
df_clean = df.copy()

# Drop columns that would cause data leakage
df_clean = df_clean.drop(columns=['diagnosed_diabetes', 'diabetes_risk_score'])
print("Dropped leakage columns: diagnosed_diabetes, diabetes_risk_score")

# Define column types based on actual data
numeric_cols = ['Age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week',
                'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day',
                'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp',
                'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
                'triglycerides', 'glucose_fasting', 'glucose_postprandial',
                'insulin_level', 'hba1c']

categorical_cols = ['gender', 'ethnicity', 'education_level', 'income_level',
                    'employment_status', 'smoking_status', 'family_history_diabetes',
                    'hypertension_history', 'cardiovascular_history']

# Fill numeric missing values with median
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"Filled '{col}' with median: {median_val:.2f}")

# Fill categorical missing values with mode
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"Filled '{col}' with mode: {mode_val}")

print("\nMissing values remaining:", df_clean.isnull().sum().sum())
print("Shape after cleaning:", df_clean.shape)

Dropped leakage columns: diagnosed_diabetes, diabetes_risk_score

Missing values remaining: 0
Shape after cleaning: (97297, 29)


In [8]:
df_encoded = df_clean.copy()

# Binary columns - label encode
binary_cols = ['gender', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history']

le = LabelEncoder()
for col in binary_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])
    print(f"Label encoded '{col}': {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Multi-category columns - one-hot encode
multi_cols = ['ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status']

for col in multi_cols:
    dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=True)
    df_encoded = pd.concat([df_encoded.drop(columns=[col]), dummies], axis=1)
    print(f"One-hot encoded '{col}'")

print("\nShape after encoding:", df_encoded.shape)

Label encoded 'gender': {'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2)}
Label encoded 'family_history_diabetes': {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}
Label encoded 'hypertension_history': {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}
Label encoded 'cardiovascular_history': {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}
One-hot encoded 'ethnicity'
One-hot encoded 'education_level'
One-hot encoded 'income_level'
One-hot encoded 'employment_status'
One-hot encoded 'smoking_status'

Shape after encoding: (97297, 40)


In [10]:
print("Diabetes stage classes:", df_encoded['diabetes_stage'].unique())

target_le = LabelEncoder()
df_encoded['diabetes_stage_encoded'] = target_le.fit_transform(df_encoded['diabetes_stage'])

print("\nTarget encoding mapping:")
for cls, enc in zip(target_le.classes_, target_le.transform(target_le.classes_)):
    print(f"  {cls} → {enc}")

os.makedirs('../artifacts', exist_ok=True)
with open('../artifacts/target_encoder.pkl', 'wb') as f:
    pickle.dump(target_le, f)
print("\nTarget encoder saved.")

Diabetes stage classes: ['Type 2' 'No Diabetes' 'Pre-Diabetes' 'Gestational' 'Type 1']

Target encoding mapping:
  Gestational → 0
  No Diabetes → 1
  Pre-Diabetes → 2
  Type 1 → 3
  Type 2 → 4

Target encoder saved.


In [13]:
df_no_outliers = df_encoded.copy()

for col in numeric_cols:
    Q1 = df_no_outliers[col].quantile(0.25)
    Q3 = df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df_no_outliers[col] < lower) | (df_no_outliers[col] > upper)).sum()
    if outliers > 0:
        df_no_outliers[col] = df_no_outliers[col].clip(lower, upper)
        print(f"'{col}': capped {outliers} outliers")

print("\nShape after outlier handling:", df_no_outliers.shape)

'alcohol_consumption_per_week': capped 443 outliers
'physical_activity_minutes_per_week': capped 3115 outliers
'diet_score': capped 325 outliers
'sleep_hours_per_day': capped 870 outliers
'screen_time_hours_per_day': capped 302 outliers
'bmi': capped 729 outliers
'waist_to_hip_ratio': capped 265 outliers
'systolic_bp': capped 517 outliers
'diastolic_bp': capped 717 outliers
'heart_rate': capped 836 outliers
'cholesterol_total': capped 296 outliers
'hdl_cholesterol': capped 555 outliers
'ldl_cholesterol': capped 340 outliers
'triglycerides': capped 291 outliers
'glucose_fasting': capped 722 outliers
'glucose_postprandial': capped 617 outliers
'insulin_level': capped 322 outliers
'hba1c': capped 599 outliers

Shape after outlier handling: (97297, 41)


In [14]:
X = df_no_outliers.drop(columns=['diabetes_stage', 'diabetes_stage_encoded'])
y = df_no_outliers['diabetes_stage_encoded']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Training set:   {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]:,} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\nClass distribution in training set:")
print(pd.Series(y_train).value_counts().sort_index())

Training set:   68,107 rows (70.0%)
Validation set: 14,595 rows (15.0%)
Test set:       14,595 rows (15.0%)

Class distribution in training set:
diabetes_stage_encoded
0      187
1     5416
2    21709
3       82
4    40713
Name: count, dtype: int64


In [15]:
scaler = StandardScaler()
feature_cols = X_train.columns.tolist()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete.")
print("Mean of scaled training features (should be ~0):", 
      X_train_scaled[numeric_cols].mean().mean().round(4))

Scaling complete.
Mean of scaled training features (should be ~0): -0.0


In [17]:
os.makedirs('../data/processed', exist_ok=True)

# Unscaled (for tree models - Role 3)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

# Scaled (for K-Means - Role 4)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_val_scaled.to_csv('../data/processed/X_val_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)

# Target labels
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler and feature list
with open('../artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("All files saved!")
for f in os.listdir('../data/processed'):
    print(f"  {f}")

All files saved!
  X_test.csv
  X_test_scaled.csv
  X_train.csv
  X_train_scaled.csv
  X_val.csv
  X_val_scaled.csv
  y_test.csv
  y_train.csv
  y_val.csv


In [18]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Original dataset shape:     {df.shape}")
print(f"After preprocessing shape:  {df_no_outliers.shape}")
print(f"Features used:              {len(feature_cols)}")
print(f"Target classes:             {list(target_le.classes_)}")
print(f"\nSplit sizes:")
print(f"  Train:      {len(X_train):,} rows")
print(f"  Validation: {len(X_val):,} rows")
print(f"  Test:       {len(X_test):,} rows")
print("\nArtifacts saved: scaler.pkl, target_encoder.pkl, feature_columns.pkl")

PREPROCESSING SUMMARY
Original dataset shape:     (97297, 31)
After preprocessing shape:  (97297, 41)
Features used:              39
Target classes:             ['Gestational', 'No Diabetes', 'Pre-Diabetes', 'Type 1', 'Type 2']

Split sizes:
  Train:      68,107 rows
  Validation: 14,595 rows
  Test:       14,595 rows

Artifacts saved: scaler.pkl, target_encoder.pkl, feature_columns.pkl
